Feature Engineering, Feature Selection & Data Leakage


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Same project structure used by the previous team notebooks.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hotel-booking-cancellation')
except ImportError:
    ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'data/processed/hotel_bookings_cleaned.csv').exists()
    )

CLEAN_PATH = ROOT / 'data/processed/hotel_bookings_cleaned.csv'
FEATURE_PATH = ROOT / 'data/processed/hotel_bookings_features.csv'
REPORT_DIR = ROOT / 'reports/member4'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH)

# Member 2 intentionally converted agent IDs to text.
if 'agent' in df.columns:
    df['agent'] = df['agent'].astype(str)

TARGET = 'is_canceled'

assert TARGET in df.columns, f"Target '{TARGET}' is missing."
assert set(df[TARGET].dropna().unique()).issubset({0, 1}), "Target must be binary."

print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')
display(df.head())


Mounted at /content/drive
Loaded cleaned data: 87,222 rows x 31 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,No_Agent,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,No_Agent,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,No_Agent,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
handover_check = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing': df.isna().sum().values,
    'unique_values': df.nunique(dropna=True).values
})

display(handover_check)

print('Rows:', len(df))
print('Columns:', df.shape[1])
print('Remaining exact duplicates:', df.duplicated().sum())
print('Remaining missing cells:', int(df.isna().sum().sum()))


,column,dtype,missing,unique_values
0,hotel,object,0,2
1,is_canceled,int64,0,2
2,lead_time,int64,0,479
3,arrival_date_year,int64,0,3
4,arrival_date_month,object,0,12
5,arrival_date_week_number,int64,0,53
6,arrival_date_day_of_month,int64,0,31
7,stays_in_weekend_nights,int64,0,17
8,stays_in_week_nights,int64,0,33
9,adults,int64,0,14


Rows: 87222
Columns: 31
Remaining exact duplicates: 0
Remaining missing cells: 0


##Audit candidate features for leakage and timing

The decisions below deliberately separate three cases:

- **EXCLUDE — confirmed leakage:** the feature directly reveals or is tied to the later outcome.
- **EXCLUDE — timing-sensitive under our booking-time scenario:** the value may change or accumulate after the initial booking.
- **KEEP — booking-time/historical candidate:** the feature is potentially valid under the stated prediction point.

For timing-sensitive variables, the reason is documented explicitly rather than silently dropping the column.


In [3]:
feature_audit = pd.DataFrame([
    {
        'feature': 'reservation_status',
        'decision': 'EXCLUDE',
        'risk': 'Confirmed outcome leakage',
        'reason': 'The final reservation status directly reveals whether the booking was cancelled.'
    },
    {
        'feature': 'reservation_status_date',
        'decision': 'EXCLUDE',
        'risk': 'Confirmed outcome leakage',
        'reason': 'This date is associated with the eventual reservation status and occurs downstream of booking.'
    },
    {
        'feature': 'assigned_room_type',
        'decision': 'EXCLUDE',
        'risk': 'Timing-sensitive',
        'reason': 'The assigned room can differ from the originally reserved room and may only be known later.'
    },
    {
        'feature': 'booking_changes',
        'decision': 'EXCLUDE',
        'risk': 'Timing-sensitive',
        'reason': 'The number of booking changes can accumulate after the initial reservation.'
    },
    {
        'feature': 'days_in_waiting_list',
        'decision': 'EXCLUDE',
        'risk': 'Timing-sensitive',
        'reason': 'Waiting-list duration can develop after the original booking and may not be known at initial scoring.'
    },
    {
        'feature': 'reserved_room_type',
        'decision': 'KEEP',
        'risk': 'Booking-time candidate',
        'reason': 'The room type requested by the customer is part of the reservation information.'
    },
    {
        'feature': 'lead_time',
        'decision': 'KEEP',
        'risk': 'Booking-time candidate',
        'reason': 'Lead time is defined from booking to arrival and is available when the reservation is created.'
    },
    {
        'feature': 'hotel',
        'decision': 'KEEP',
        'risk': 'Booking-time candidate',
        'reason': 'The booked hotel is known when the reservation is made.'
    },
    {
        'feature': 'market_segment',
        'decision': 'KEEP',
        'risk': 'Booking-time candidate',
        'reason': 'Booking source/segment information is available as part of the reservation record.'
    },
    {
        'feature': 'customer_type',
        'decision': 'KEEP',
        'risk': 'Booking-time candidate',
        'reason': 'Customer type is treated as reservation information available at scoring time.'
    },
    {
        'feature': 'previous_cancellations',
        'decision': 'KEEP',
        'risk': 'Historical cutoff required',
        'reason': 'Valid only when it represents customer history before the current booking.'
    },
    {
        'feature': 'previous_bookings_not_canceled',
        'decision': 'KEEP',
        'risk': 'Historical cutoff required',
        'reason': 'Valid only when it represents customer history before the current booking.'
    },
    {
        'feature': 'is_repeated_guest',
        'decision': 'KEEP',
        'risk': 'Historical cutoff required',
        'reason': 'Valid when repeat-guest status is based only on history before the current booking.'
    },
    {
        'feature': 'deposit_type',
        'decision': 'KEEP WITH ASSUMPTION',
        'risk': 'Snapshot timing must be verified',
        'reason': 'Member 1 found a strong association; keep only if deposit type is fixed/known at the booking-time snapshot.'
    },
    {
        'feature': 'adr',
        'decision': 'KEEP WITH ASSUMPTION',
        'risk': 'Snapshot timing must be verified',
        'reason': 'Use only as the booking-time ADR, not a value revised using later stay information.'
    },
    {
        'feature': 'required_car_parking_spaces',
        'decision': 'KEEP WITH ASSUMPTION',
        'risk': 'Snapshot timing must be verified',
        'reason': 'Keep if this is the parking request recorded when the booking is scored.'
    },
    {
        'feature': 'total_of_special_requests',
        'decision': 'KEEP WITH ASSUMPTION',
        'risk': 'Snapshot timing must be verified',
        'reason': 'Keep if this count represents requests already known at the scoring point.'
    },
])

# Show only rows for columns that exist in the cleaned dataset.
feature_audit = feature_audit[feature_audit['feature'].isin(df.columns)].reset_index(drop=True)
display(feature_audit)

feature_audit.to_csv(REPORT_DIR / 'feature_timing_leakage_audit.csv', index=False)


,feature,decision,risk,reason
0,reservation_status,EXCLUDE,Confirmed outcome leakage,The final reservation status directly reveals ...
1,reservation_status_date,EXCLUDE,Confirmed outcome leakage,This date is associated with the eventual rese...
2,assigned_room_type,EXCLUDE,Timing-sensitive,The assigned room can differ from the original...
3,booking_changes,EXCLUDE,Timing-sensitive,The number of booking changes can accumulate a...
4,days_in_waiting_list,EXCLUDE,Timing-sensitive,Waiting-list duration can develop after the or...
5,reserved_room_type,KEEP,Booking-time candidate,The room type requested by the customer is par...
6,lead_time,KEEP,Booking-time candidate,Lead time is defined from booking to arrival a...
7,hotel,KEEP,Booking-time candidate,The booked hotel is known when the reservation...
8,market_segment,KEEP,Booking-time candidate,Booking source/segment information is availabl...
9,customer_type,KEEP,Booking-time candidate,Customer type is treated as reservation inform...


### 4.1 Direct demonstration of confirmed leakage

If `reservation_status` almost directly identifies `is_canceled`, using it would make model performance look unrealistically strong while making the model unusable for a real booking-time prediction.

This table is descriptive evidence only; no model is trained.


In [4]:
if 'reservation_status' in df.columns:
    status_target = pd.crosstab(
        df['reservation_status'],
        df[TARGET],
        margins=True
    )
    display(status_target)


is_canceled,0,1,All
reservation_status,,,
Canceled,0,22996,22996
Check-Out,63213,0,63213
No-Show,0,1013,1013
All,63213,24009,87222


## 5. Remove only the approved leakage/timing-sensitive fields

Under the team's booking-time scenario, this notebook excludes:

- `reservation_status`
- `reservation_status_date`
- `assigned_room_type`
- `booking_changes`
- `days_in_waiting_list`

No column is removed merely because it has weak correlation with the target.


In [5]:
FEATURE_EXCLUSIONS = [
    'reservation_status',
    'reservation_status_date',
    'assigned_room_type',
    'booking_changes',
    'days_in_waiting_list',
]

FEATURE_EXCLUSIONS = [c for c in FEATURE_EXCLUSIONS if c in df.columns]

features_df = df.drop(columns=FEATURE_EXCLUSIONS).copy()

print('Excluded features:', FEATURE_EXCLUSIONS)
print('Shape before exclusions:', df.shape)
print('Shape after exclusions:', features_df.shape)


Excluded features: ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'days_in_waiting_list']
Shape before exclusions: (87222, 31)
Shape after exclusions: (87222, 26)


## 6. Feature engineering

Feature engineering combines existing booking-time information into variables with clearer business meaning.

The engineered features below were motivated by Member 1's exploratory analysis and are now formally created by Member 4.

### 6.1 `total_stay`

`total_stay = stays_in_weekend_nights + stays_in_week_nights`

This represents the complete planned length of stay.

### 6.2 `total_guests`

`total_guests = adults + children + babies`

Member 2 already resolved the four missing `children` values, so total guest count can now be calculated without the uncertainty present in Member 1's raw-data EDA.

### 6.3 `is_family`

A binary indicator equal to 1 when the booking contains at least one child or baby.

These are derived only from information already present in the booking record.


In [6]:
required_for_engineering = [
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies'
]

missing_required = [c for c in required_for_engineering if c not in features_df.columns]
assert not missing_required, f'Missing required source columns: {missing_required}'

features_df['total_stay'] = (
    features_df['stays_in_weekend_nights']
    + features_df['stays_in_week_nights']
)

features_df['total_guests'] = (
    features_df['adults']
    + features_df['children']
    + features_df['babies']
)

features_df['is_family'] = (
    (features_df['children'] > 0)
    | (features_df['babies'] > 0)
).astype(int)

ENGINEERED_FEATURES = ['total_stay', 'total_guests', 'is_family']

display(features_df[ENGINEERED_FEATURES].head())


,total_stay,total_guests,is_family
0,0,2.0,0
1,0,2.0,0
2,1,1.0,0
3,1,1.0,0
4,2,2.0,0


## 7. Validate engineered features

These checks verify that the derived variables are logically consistent.

They do **not** evaluate predictive model performance.


In [7]:
assert (features_df['total_stay'] >= 0).all(), 'Negative total stay found.'
assert (features_df['total_guests'] > 0).all(), 'Zero/negative total guests found after Member 2 cleaning.'
assert set(features_df['is_family'].unique()).issubset({0, 1}), 'is_family must be binary.'

engineering_summary = features_df[ENGINEERED_FEATURES].describe().T
display(engineering_summary)

print('Family bookings:', int(features_df['is_family'].sum()))
print('Non-family bookings:', int((features_df['is_family'] == 0).sum()))


,count,mean,std,min,25%,50%,75%,max
total_stay,87222.0,3.628672,2.742888,0.0,2.0,3.0,5.0,69.0
total_guests,87222.0,2.029167,0.790139,1.0,2.0,2.0,2.0,55.0
is_family,87222.0,0.104366,0.305736,0.0,0.0,0.0,0.0,1.0


Family bookings: 9103
Non-family bookings: 78119


## 8. Review redundancy created by engineered features

Engineered variables intentionally overlap with their source variables:

- `total_stay` comes from weekday and weekend nights.
- `total_guests` comes from adults, children, and babies.
- `is_family` also uses children/babies.

This does **not** automatically mean the source variables should be deleted.

Keeping both source and aggregate variables preserves potentially useful detail for tree-based models, while later model validation can determine whether the extra representation helps. Feature removal should not be based only on a simple correlation coefficient.


In [8]:
redundancy_groups = {
    'stay_information': [
        'stays_in_weekend_nights',
        'stays_in_week_nights',
        'total_stay'
    ],
    'guest_information': [
        'adults',
        'children',
        'babies',
        'total_guests',
        'is_family'
    ]
}

for group, cols in redundancy_groups.items():
    available = [c for c in cols if c in features_df.columns]
    print(f'{group}: {available}')

numeric_review = [
    c for c in [
        'lead_time', 'adr', 'previous_cancellations',
        'previous_bookings_not_canceled',
        'required_car_parking_spaces',
        'total_of_special_requests',
        'total_stay', 'total_guests', 'is_family'
    ]
    if c in features_df.columns
]

corr_review = features_df[numeric_review + [TARGET]].corr(numeric_only=True)[TARGET]
corr_review = corr_review.drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)

display(corr_review.to_frame('pearson_r_with_is_canceled'))


stay_information: ['stays_in_weekend_nights', 'stays_in_week_nights', 'total_stay']
guest_information: ['adults', 'children', 'babies', 'total_guests', 'is_family']


,pearson_r_with_is_canceled
lead_time,0.184494
required_car_parking_spaces,-0.184477
adr,0.127210
total_of_special_requests,-0.120823
total_guests,0.098946
total_stay,0.085407
previous_bookings_not_canceled,-0.052177
previous_cancellations,0.051497
is_family,0.051406


## 9. Feature-selection decision

For Stage 1, feature selection is primarily based on:

1. **availability at the prediction point,**
2. **absence of direct outcome leakage,**
3. **data quality established by Member 2,**
4. **business meaning and evidence from Member 1's EDA,**
5. **avoiding arbitrary removal based only on univariate correlation.**

Therefore:

- confirmed leakage and timing-sensitive post-booking fields are excluded;
- engineered features are added;
- the remaining booking-time/historical variables are retained for the modelling stage;
- no model-based feature importance is used here because that would belong to Stage 2 modelling/evaluation.

`arrival_date_year` is retained. The dataset covers different arrival periods, and the previous notebooks did not establish a Stage 1 reason to remove it. The team should later consider time-based validation/generalisation when interpreting model results.


In [9]:
FINAL_FEATURES = [
    c for c in features_df.columns
    if c != TARGET
]

selection_table = pd.DataFrame({
    'feature': df.columns
})

def selection_decision(feature):
    if feature == TARGET:
        return 'TARGET'
    if feature in FEATURE_EXCLUSIONS:
        return 'EXCLUDE'
    return 'KEEP'

selection_table['decision'] = selection_table['feature'].map(selection_decision)

# Add engineered features to the decision table.
engineered_rows = pd.DataFrame({
    'feature': ENGINEERED_FEATURES,
    'decision': ['ENGINEERED / KEEP'] * len(ENGINEERED_FEATURES)
})

selection_table = pd.concat([selection_table, engineered_rows], ignore_index=True)

display(selection_table)
print('Final predictor count:', len(FINAL_FEATURES))


,feature,decision
0,hotel,KEEP
1,is_canceled,TARGET
2,lead_time,KEEP
3,arrival_date_year,KEEP
4,arrival_date_month,KEEP
5,arrival_date_week_number,KEEP
6,arrival_date_day_of_month,KEEP
7,stays_in_weekend_nights,KEEP
8,stays_in_week_nights,KEEP
9,adults,KEEP


Final predictor count: 28


## 10. Final leakage safety checks

The target must remain available for supervised learning, but it must never appear inside the predictor list.

Confirmed leakage fields must also be absent from the final predictors.


In [10]:
assert TARGET not in FINAL_FEATURES, 'Target leaked into predictors.'

for col in FEATURE_EXCLUSIONS:
    assert col not in FINAL_FEATURES, f'Excluded feature still present: {col}'

for col in ENGINEERED_FEATURES:
    assert col in FINAL_FEATURES, f'Engineered feature missing: {col}'

print('Leakage safety checks passed.')
print('Target:', TARGET)
print('Final predictors:', len(FINAL_FEATURES))


Leakage safety checks passed.
Target: is_canceled
Final predictors: 28


## 11. Save outputs

`data/processed/hotel_bookings_features.csv`


In [11]:
handover_columns = FINAL_FEATURES + [TARGET]
handover_df = features_df[handover_columns].copy()

handover_df.to_csv(FEATURE_PATH, index=False)

print('Saved feature dataset:', FEATURE_PATH)
print('Final handover shape:', handover_df.shape)

Saved feature dataset: /content/drive/MyDrive/hotel-booking-cancellation/data/processed/hotel_bookings_features.csv
Final handover shape: (87222, 29)
